<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/shared_analysis/shared_analysis_roundtrip.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# Shared-Analysis Round-Trip: publish & replicate attribution-graph steering

This notebook demonstrates end-to-end shared analysis: one user generates attribution-graph
sign-aware feature-steering results; a second user replicates those results from the published
artifact with the expensive attribution-graph generation skipped entirely. Only the cheap steering forward pass re-runs from the stored graph.

1. Generate locally: the `it.intervention_from_concept` pipeline produces the reference
   results and an `AnalysisStore`.
2. Push:  the store publishes as a Hub *dataset* repo with a machine-written
   `it_artifact.json` envelope (parquet interchange; the local format stays Arrow).
3. Clean:  the local analysis state is deleted; only small reference snapshots remain.
4. Replicate:  `it.hub.pull_analysis_store` fetches the artifact, the interpretune formatter
   re-attaches from the envelope, and the named analysis backend hydrates steering-capable
   graphs; the steering results are reproduced and compared against the reference.
5. Optional coda:  a small provenance iteration re-uploads to the same repo: the artifact's
   `identity` survives, its content fingerprint tracks the change.


In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
BACKEND = "nnsight"  # circuit-tracer backend: "nnsight" or "transformerlens"
CONCEPT_PROMPT = "Is orange a color or a fruit? Answer with one word: Color or Fruit. orange ->"
CONCEPT_TARGET_TOKENS = ["Fruit", "Color"]
FEATURE_SELECTION_TOP_N = 5
FEATURE_SELECTION_MIN_LAYER = 10
FEATURE_SELECTION_SCORE_SIGN = "any"
INTERVENTION_SCALE_FACTOR = 20.0
REGISTRY_KEY = "rte_demo.gemma2.circuit_tracer"  # hub component configuration key (model + backend)
MODEL_NAME = "gemma-2-2b"
TRANSCODER_SET = "gemma"
NEURONPEDIA_MODEL_ID = "gemma-2-2b"  # public dashboard substrate: feature indices link to neuronpedia.org
NEURONPEDIA_SOURCE_SET = "gemmascope-transcoder-16k"
CHAT_FORMAT_PROMPT = False

# -- The shared-analysis artifact -----------------------------------------------------------------
# Set this to YOUR OWN `<org>/<repo>` (any org you can write to). `speediedan/...` is only the
# maintainer's validation target -- nothing about the workflow is specific to it.
ARTIFACT_REPO_ID = "speediedan/ct-concept-steering-demo"
ARTIFACT_PRIVATE = True
HF_TOKEN_ENV = "IT_HF_TOKEN"  # env var holding a WRITE token for ARTIFACT_REPO_ID's org
RUN_REUPLOAD_CODA = True  # step 5: provenance-iteration re-upload (identity must survive)

In [3]:
# @title Imports { display-mode: "form" }
import os

import torch  # noqa: F401

import interpretune.analysis  # noqa: F401  # ensure op wrappers are registered
from interpretune.analysis import analysis_store_from_batches
from interpretune.analysis.backends import FeatureSelectionSpec
from interpretune.analysis.ops.base import AnalysisBatch
from it_examples.utils.nb_ui_utils import (  # noqa: F401
    display_steering_results,
    display_target_gap,
)

## 1. Generate locally:  the reference results

In [4]:
# @title 1: Session construction { display-mode: "form" }
from pathlib import Path

from dotenv import load_dotenv

import interpretune as it
from it_examples.seeds import ensure_local_seeds
from interpretune import ITSession, ITSessionConfig

for _env_candidate in (Path.cwd() / ".env", Path.home() / "repos" / "interpretune" / ".env"):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

ensure_local_seeds()  # idempotent, offline: seed publish sources -> local components cache
base_itdm_cfg, base_it_cfg, dm_cls, m_cls = it.hub.load("speediedan/rte", REGISTRY_KEY)
base_it_cfg.circuit_tracer_cfg.backend = BACKEND
if TRANSCODER_SET:
    base_it_cfg.circuit_tracer_cfg.transcoder_set = TRANSCODER_SET
if BACKEND == "nnsight":
    adapter_ctx = (it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer)
else:
    adapter_ctx = (it.Adapter.core, it.Adapter.transformer_lens, it.Adapter.circuit_tracer)
session_cfg = ITSessionConfig(
    adapter_ctx=adapter_ctx,
    datamodule_cfg=base_itdm_cfg,
    module_cfg=base_it_cfg,
    datamodule_cls=dm_cls,
    module_cls=m_cls,
)
it_session = ITSession(session_cfg)
it.it_init(**it_session)
module = it_session.module
tokenizer = module.replacement_model.tokenizer
print(f"session ready: {type(module).__name__} ({MODEL_NAME} + circuit-tracer {BACKEND} backend)")

~/worktrees/it-420-basis/src/interpretune/config/shared.py:325: Could not find an auto-composition for <class 'interpretune.config.module.ITConfig'> that supports all of the following kwargs: {'tl_cfg': ITLensFromPretrainedNoProcessingConfig(move_to_device=True, default_padding_side='left', use_bridge=False, model_name='gemma-2-2b', fold_ln=False, center_writing_weights=False, center_unembed=False, refactor_factored_attn_matrices=False, checkpoint_index=None, checkpoint_value=None, hf_model=None, device='cuda', n_devices=1, tokenizer=None, fold_value_biases=False, default_prepend_bos=True, dtype='float32'), 'circuit_tracer_cfg': CircuitTracerConfig(backend='transformerlens', model_name=None, transcoder_set='gemma', dtype=torch.bfloat16, max_n_logits=10, desired_logit_prob=0.95, batch_size=256, max_feature_nodes=8192, offload='cpu', lazy_encoder=None, lazy_decoder=True, verbose=True, default_node_threshold=0.8, default_edge_threshold=0.98, save_graphs=True, graph_output_dir=None, analys

[INFO] interpretune.utils.logging: Loading ReplacementModel with backend: nnsight


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

...

[INFO] interpretune.utils.logging: Setting up datamodule: InterpretunableDataModule


[INFO] interpretune.utils.logging: Setting up model: InterpretunableModule


[INFO] interpretune.utils.logging: initializing optimizers and schedulers: InterpretunableModule


[INFO] interpretune.utils.logging: Input gradient requirements handled by circuit tracer internally.


session ready: InterpretunableModule (gemma-2-2b + circuit-tracer nnsight backend)


In [5]:
# @title 2: Run the pipeline; capture the reference; build the store { display-mode: "form" }
from interpretune.config import AnalysisCfg, init_analysis_cfgs

module.analysis_cfg = AnalysisCfg(target_op=it.intervention_from_concept, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module, [module.analysis_cfg])

fruits = ["apple", "banana", "grape", "peach"]
colors = ["red", "blue", "green", "yellow"]
if CHAT_FORMAT_PROMPT:
    from it_examples.examples.prompt_configs.prompt_configs import GemmaPromptConfig

    prompt = GemmaPromptConfig().apply_chat_template_fn(
        tokenizer, CONCEPT_PROMPT, tokenize=False, add_generation_prompt=True
    )
else:
    prompt = CONCEPT_PROMPT

ct_cfg = module.it_cfg.circuit_tracer_cfg
ct_cfg.intervention_sign_aware_scale = True
ct_cfg.intervention_max_influence_norm_scale = True
ct_cfg.intervention_value_source = "top_feature_activation_values"

selection_spec = FeatureSelectionSpec(
    layer_slice=slice(FEATURE_SELECTION_MIN_LAYER, None),
    score_source="signed_influence",
    score_sign=FEATURE_SELECTION_SCORE_SIGN,
    rank_by_abs=True,
)
pipeline_results = it.intervention_from_concept(
    module,
    AnalysisBatch(
        concept_group_a=fruits,
        concept_group_b=colors,
        concept_label="Concept: Fruit - Color",
        concept_direction_mode="paired_rejection",
        concept_basis="embed",
        prompts=[prompt],
    ),
    None,
    0,
    top_n=FEATURE_SELECTION_TOP_N,
    intervention_scale_factor=INTERVENTION_SCALE_FACTOR,
    feature_selection=selection_spec,
)
steering = display_steering_results(
    pipeline_results,
    tokenizer,
    CONCEPT_TARGET_TOKENS,
    neuronpedia_model=NEURONPEDIA_MODEL_ID,
    neuronpedia_set=NEURONPEDIA_SOURCE_SET,
)

# the store IS the shareable artifact: the pipeline results serialized by the op's own schema
store = analysis_store_from_batches(
    module, [pipeline_results], op=it.intervention_from_concept, analysis_backend="circuit_tracer"
)
# REFERENCE snapshots (plain python; everything else local is deleted in step 3)
reference_row = {k: v for k, v in store.dataset.with_format(None)[0].items()}
reference_columns = sorted(store.dataset.column_names)
print(f"reference captured: {len(reference_columns)} columns, 1 row")

Phase 0: Precomputing activations and vectors


Precomputation completed in 1.91s


Found 18348 active features


Phase 1: Running forward pass


Forward pass completed in 0.54s


...


Feature attributions completed in 4.29s


Attribution completed in 10.28s


#,Node,Sign,|Score|
1,"(25, 19, 16131)",−,2.11e-08
2,"(24, 19, 13277)",−,1.97e-08
3,"(24, 19, 5999)",+,1.08e-08
4,"(24, 19, 3865)",+,5.31e-09
5,"(25, 19, 13210)",+,5.17e-09


Token,Pre prob,Post prob,Pre logit,Post logit,Δ
Fruit,2.317%,1.868%,25.1250,29.3750,+4.2500
Color,15.106%,1.03e-05,27.0000,21.8750,-5.1250
Gap (Fruit − Color),,,-1.8750,+7.5000,+9.3750


reference captured: 41 columns, 1 row


## 2. Push the store to the Hub (parquet interchange + envelope + generated card)

In [6]:
# @title 3: Push { display-mode: "form" }
token = os.environ[HF_TOKEN_ENV]
pushed_revision = it.hub.push_analysis_store(
    store,
    ARTIFACT_REPO_ID,
    private=ARTIFACT_PRIVATE,
    token=token,
    provenance={"generated_by": "intervention_from_concept", "model_name": MODEL_NAME, "backend": BACKEND},
)
print(f"pushed {ARTIFACT_REPO_ID} @ {pushed_revision}")

it_artifact.json:   0%|          | 0.00/1.81k [00:00<?, ?B/s]

Setting num_proc from 1 back to 1 for the validation split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

pushed speediedan/ct-concept-steering-demo @ 74700916bdeb59199b0f6f172c2b06ec3940243f


## 3. Clean the local analysis state

Everything the pipeline produced locally is deleted:  what follows must come from the Hub.

In [7]:
# @title 4: Clean { display-mode: "form" }
import gc
import shutil

local_save_dir = getattr(getattr(module.analysis_cfg, "output_store", None), "save_dir", None)
del store, pipeline_results
gc.collect()
if local_save_dir and Path(local_save_dir).exists():
    shutil.rmtree(local_save_dir)
    print(f"removed local analysis output: {local_save_dir}")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("local analysis state cleaned -- only the reference snapshots remain in memory")

removed local analysis output: /tmp/20260915_085927_default/analysis_datasets
local analysis state cleaned -- only the reference snapshots remain in memory


## 4. Replicate from the Hub artifact:  no pipeline re-run

`pull_analysis_store` fetches envelope-first (revision-pinned), re-attaches the interpretune
formatter from the envelope, and resolves the named analysis backend (`circuit_tracer`) so
row access hydrates steering-capable `Graph` objects. We verify three things:

1. Transport fidelity:  every stored column round-trips the parquet interchange intact.
2. Hydration:  the pulled store yields a live attribution `Graph` via the backend seam.
3. Steering replication:  the graph computation is skipped; `it.intervention_from_features`
   re-runs only the (cheap) steering forward pass from the stored graph and reproduces the
   reference steering outcome. The compute the second user avoids is the expensive
   attribution analysis.

In [8]:
# @title 5: Pull + verify { display-mode: "form" }
import numpy as np

pulled = it.hub.pull_analysis_store(ARTIFACT_REPO_ID, token=token)
envelope = it.hub.describe_analysis_store(ARTIFACT_REPO_ID)
print(f"artifact identity: {envelope['identity']['store_id']} (created {envelope['identity']['created_utc']})")
print(f"provenance: {envelope['provenance']}")

# 1) transport fidelity: every column byte-faithful through parquet
pulled_row = {k: v for k, v in pulled.dataset.with_format(None)[0].items()}
assert sorted(pulled.dataset.column_names) == reference_columns
mismatched = []
for col in reference_columns:
    ref, got = reference_row[col], pulled_row[col]
    equal = np.array_equal(np.asarray(ref, dtype=object), np.asarray(got, dtype=object))
    if not equal:
        mismatched.append(col)
assert not mismatched, f"columns changed in transport: {mismatched}"
print(f"transport fidelity: {len(reference_columns)}/{len(reference_columns)} columns identical")

# 2) hydration via the named backend seam (no generation pipeline involved)
hydrated = pulled[0]
graph = hydrated["attribution_graph"]
print(f"hydrated graph: {type(graph).__name__} ({graph.active_features.shape[0]} active features)")

# 3) steering replication: ONLY the intervention forward pass re-runs, from hydrated state
# the prompt itself travels IN the artifact (the graph's input_string) -- the second user
# needs nothing beyond the hub repo to re-run the steering forward
replicated = it.intervention_from_features(
    module,
    AnalysisBatch(**{k: v for k, v in hydrated.items()}, prompts=[graph.input_string]),
    None,
    0,
    intervention_scale_factor=INTERVENTION_SCALE_FACTOR,
)
replicated_steering = display_steering_results(
    replicated,
    tokenizer,
    CONCEPT_TARGET_TOKENS,
    neuronpedia_model=NEURONPEDIA_MODEL_ID,
    neuronpedia_set=NEURONPEDIA_SOURCE_SET,
)
print("steering replicated: graph computation skipped; steering ran from the stored graph")

it_artifact.json:   0%|          | 0.00/1.85k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Generating validation split: 0 examples [00:00, ? examples/s]

artifact identity: aadcd8d6-3892-475e-9f8f-1cb21cde2305 (created 2026-08-11T23:35:47+00:00)
provenance: {'backend': 'nnsight', 'content_fingerprint': 'cdef900891af7208e09bfdf3e7b9a646d5c7f3d2447c33641b16dd59f4dddcea', 'generated_by': 'intervention_from_concept', 'interpretune_version': '0.1.0.dev455+ga87f3eb4e', 'model_name': 'gemma-2-2b'}


transport fidelity: 41/41 columns identical


hydrated graph: Graph (18348 active features)


#,Node,Sign,|Score|
1,"(25, 19, 16131)",−,2.11e-08
2,"(24, 19, 13277)",−,1.97e-08
3,"(24, 19, 5999)",+,1.08e-08
4,"(24, 19, 3865)",+,5.31e-09
5,"(25, 19, 13210)",+,5.17e-09


Token,Pre prob,Post prob,Pre logit,Post logit,Δ
Fruit,2.317%,1.868%,25.1250,29.3750,+4.2500
Color,15.106%,1.03e-05,27.0000,21.8750,-5.1250
Gap (Fruit − Color),,,-1.8750,+7.5000,+9.3750


steering replicated: graph computation skipped; steering ran from the stored graph


## 5. Optional coda:  share an iteration

Re-uploading after a change keeps the artifact's identity (`store_id` never rewritten) while
its provenance:  including the content fingerprint,  refreshes.

In [9]:
# @title 6: Re-upload coda { display-mode: "form" }
if RUN_REUPLOAD_CODA:
    original_identity = envelope["identity"]
    coda_revision = it.hub.push_analysis_store(
        pulled,
        ARTIFACT_REPO_ID,
        private=ARTIFACT_PRIVATE,
        token=token,
        provenance={"generated_by": "intervention_from_concept", "iteration_note": "round-trip coda re-upload"},
    )
    refreshed = it.hub.pull_analysis_store(ARTIFACT_REPO_ID, token=token)  # refresh the cached envelope
    coda_envelope = it.hub.describe_analysis_store(ARTIFACT_REPO_ID)
    assert coda_envelope["identity"] == original_identity, "identity must survive re-upload"
    print(f"re-uploaded @ {coda_revision}; identity preserved: {coda_envelope['identity']['store_id']}")
    print(f"iteration note: {coda_envelope['provenance'].get('iteration_note')}")
else:
    print("coda skipped (RUN_REUPLOAD_CODA=False)")

it_artifact.json:   0%|          | 0.00/1.85k [00:00<?, ?B/s]

Setting num_proc from 1 back to 1 for the validation split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

it_artifact.json:   0%|          | 0.00/1.85k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Generating validation split: 0 examples [00:00, ? examples/s]

re-uploaded @ abfed12ecd66e39420ac338908acc033fdf4f220; identity preserved: aadcd8d6-3892-475e-9f8f-1cb21cde2305
iteration note: round-trip coda re-upload


## Summary

- The `it.intervention_from_concept` results became a portable artifact: parquet interchange +
  `it_artifact.json` envelope + generated dataset card, published with `it.hub.push_analysis_store`.
- After deleting all local analysis state, `it.hub.pull_analysis_store` reproduced the exact stored
  columns, hydrated a steering-capable attribution `Graph` through the named backend seam, and
  `it.intervention_from_features` replicated the steering outcome:  the attribution-graph
  generation never re-ran; only the steering forward pass did, from the stored graph.
- Re-uploading kept the artifact's identity while its content fingerprint tracked the change.
